Run before everything

In [ ]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Part1

In [ ]:
import time

def part1run():
    print("Part 1 Started")
    
    # --- 1. Recognise Red (5 pts) ---
    while True:
        color = got.get_color_total_info()
        if color and color[0] == "Red":
            got.screen_display_background(3) # Red screen for judge
            got.screen_print("RED: STOP")
            time.sleep(0.1)
        else:
            break # Move to next phase when Red is gone
            
    # --- 2. Recognise Green & Move (5 pts) ---
    while True:
        color = got.get_color_total_info()
        if color and color[0] == "Green":
            got.screen_display_background(6) # Green screen for judge
            got.screen_print("GREEN: GO")
            time.sleep(0.5)
            # Move off start point
            got.mecanum_move_xyz(0, 15, 0) 
            time.sleep(2.0)
            got.mecanum_stop()
            break
        time.sleep(0.1)

    # --- 3. Recognise AprilTag & Centralise (20 pts) ---
    got.screen_display_background(0)
    while True:
        tag = got.get_apriltag_total_info()
        if tag:
            got.screen_print("TAG FOUND")
            # Uses distance and gap to perfectly align
            AP_centralization_approaching(distance=0.14, gap=15, fwd_spd=7, strafe_spd=8)
            
            # Final micro-adjustment for the arm reach
            got.mecanum_move_xyz(0, 5, 0)
            time.sleep(0.4)
            got.mecanum_stop()
            
            # --- 4. Pick up Token (30 pts) ---
            # Criterion: Held in clamp and off the ground
            pick_up() 
            got.screen_print("TOKEN SECURED")
            break

        # Search rotation if tag not seen
        got.mecanum_turn_speed(turn=2, speed=10) 
        time.sleep(0.1)

# Run Part 1
try:
    part1run()
except KeyboardInterrupt:
    got.mecanum_stop()